# 02 · Quantum kernel vs RBF kernel và geometric difference

**Mục tiêu:** tái hiện ý tưởng *geometric difference* trong Huang et al. (2021), *Power of data in quantum machine learning* (arXiv:2011.01938), trên dữ liệu Iris.

Câu hỏi: kernel lượng tử nhìn dữ liệu **khác** kernel cổ điển đến mức nào? Nếu gần như giống nhau, quantum không có cơ hội vượt trội.

**Phát hiện đáng chú ý:** feature map ban đầu có cổng `CNOT` bị tự triệt tiêu trong mạch kernel, khiến "quantum kernel" thực chất là một công thức cổ điển. Phần 3 chứng minh điều này.

## 1. Thư viện

In [1]:
import pennylane as qml
from pennylane import numpy as np
from sklearn.datasets import load_iris
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from scipy.linalg import sqrtm

iris = load_iris()

## 2. Feature map và quantum kernel

Dùng 2 đặc trưng đầu (chiều dài và chiều rộng đài hoa), mỗi đặc trưng vào 1 qubit.

So sánh hai feature map:
- **A (bản gốc):** `RX(x0)`, `RX(x1)`, rồi `CNOT`.
- **B (bản sửa):** như A, thêm `RZ(x0·x1)` **sau** `CNOT` để entanglement thật sự ảnh hưởng tới kernel.

**Quantum kernel** giữa hai điểm: chạy feature map của x1, rồi chạy ngược feature map của x2, và đo xác suất quay về |00⟩. Hai điểm càng giống nhau thì xác suất càng gần 1.

In [2]:
dev = qml.device("default.qubit", wires=2)

def feature_map_A(x):
    qml.RX(x[0], wires=0)
    qml.RX(x[1], wires=1)
    qml.CNOT(wires=[0, 1])

def feature_map_B(x):
    feature_map_A(x)
    qml.RZ(x[0] * x[1], wires=1)

def make_kernel(feature_map):
    @qml.qnode(dev)
    def kernel_circuit(x1, x2):
        feature_map(x1)
        qml.adjoint(feature_map)(x2)
        return qml.probs(wires=[0, 1])
    return lambda x1, x2: kernel_circuit(x1, x2)[0]

kernel_A = make_kernel(feature_map_A)
kernel_B = make_kernel(feature_map_B)

def kernel_matrix(XA, XB, kernel):
    return np.array([[kernel(a, b) for b in XB] for a in XA])

## 3. Kiểm chứng: CNOT trong feature map A bị triệt tiêu

Trong mạch kernel, thứ tự cổng là `RX(x1) → CNOT → CNOT† → RX†(x2)`. Vì `CNOT · CNOT = I`, hai cổng ở giữa triệt tiêu nhau. Kernel A khi đó chỉ còn:

$$K_A(x, x') = \cos^2\!\left(\tfrac{x_0 - x'_0}{2}\right)\cos^2\!\left(\tfrac{x_1 - x'_1}{2}\right)$$

Đây là công thức hoàn toàn cổ điển, không có entanglement nào. Ô dưới đây kiểm tra điều đó bằng số.

In [3]:
np.random.seed(42)
idx = np.random.choice(100, 50, replace=False)
x = iris.data[idx, :2]
x = (x - x.mean(axis=0)) / x.std(axis=0)

K_A = kernel_matrix(x, x, kernel_A)
cos_formula = (np.cos((x[:, None, 0] - x[None, :, 0]) / 2) ** 2 *
               np.cos((x[:, None, 1] - x[None, :, 1]) / 2) ** 2)
print("Kernel A trùng với công thức cos cổ điển:", np.allclose(K_A, cos_formula))

Kernel A trùng với công thức cos cổ điển: True


## 4. Geometric difference g

$$g = \sqrt{\lambda_{\max}\left(\sqrt{K_Q}\,(K_C + \lambda I)^{-1}\sqrt{K_Q}\right)}$$

- `K_Q`: ma trận quantum kernel, `K_C`: ma trận RBF kernel.
- `λ = 1.1`: regularization để phép nghịch đảo ổn định. Đây là bản đơn giản hóa, công thức gốc trong Huang et al. hơi khác.
- **g gần 1:** hai kernel gần như giống nhau. **g gần √N:** kernel lượng tử nhìn dữ liệu rất khác, nên *có thể* có lợi thế.
- Dùng `eigvalsh` vì ma trận đối xứng, cho kết quả chính xác và không sinh phần ảo.

In [4]:
def geometric_difference(K_Q, K_C, lam=1.1):
    KQ_sqrt = sqrtm(K_Q).real
    M = KQ_sqrt @ np.linalg.inv(K_C + lam * np.eye(len(K_C))) @ KQ_sqrt
    M = (M + M.T) / 2  # đối xứng hóa để loại sai số làm tròn
    return np.sqrt(np.max(np.linalg.eigvalsh(M)))

print(f"{'N':>4} | {'g (A)':>7} {'g/√N (A)':>9} | {'g (B)':>7} {'g/√N (B)':>9}")
for N in [5, 20, 50]:
    np.random.seed(42)
    idx = np.random.choice(100, N, replace=False)
    xs = iris.data[idx, :2]
    xs = (xs - xs.mean(axis=0)) / xs.std(axis=0)
    K_C = rbf_kernel(xs, xs)
    gA = geometric_difference(kernel_matrix(xs, xs, kernel_A), K_C)
    gB = geometric_difference(kernel_matrix(xs, xs, kernel_B), K_C)
    print(f"{N:>4} | {gA:7.3f} {gA/np.sqrt(N):9.1%} | {gB:7.3f} {gB/np.sqrt(N):9.1%}")

   N |   g (A)  g/√N (A) |   g (B)  g/√N (B)
   5 |   0.893     39.9% |   0.905     40.5%
  20 |   1.085     24.3% |   1.096     24.5%
  50 |   1.148     16.2% |   1.187     16.8%


**Nhận xét:** g tăng theo N nhưng tỉ lệ g/√N giảm, nghĩa là càng nhiều dữ liệu, quantum kernel càng gần với RBF. Feature map B chỉ tăng g rất ít: với 2 qubit và feature map đơn giản, kernel lượng tử vẫn "nhìn" dữ liệu khá giống kernel cổ điển.

## 5. So sánh độ chính xác SVM

Dùng `SVC(kernel="precomputed")` với ma trận quantum kernel, so với SVM dùng RBF. Chia 70/30 train/test.

In [5]:
def compare_svm(class_a, class_b, name):
    mask = np.isin(iris.target, [class_a, class_b])
    X, y = iris.data[mask][:, :2], iris.target[mask]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    mean, std = X_tr.mean(axis=0), X_tr.std(axis=0)
    X_tr, X_te = (X_tr - mean) / std, (X_te - mean) / std

    print(name)
    rbf = SVC(kernel="rbf").fit(X_tr, y_tr)
    print(f"  RBF kernel:          {rbf.score(X_te, y_te):.1%}")
    for label, kern in [("Quantum kernel A", kernel_A), ("Quantum kernel B", kernel_B)]:
        clf = SVC(kernel="precomputed").fit(kernel_matrix(X_tr, X_tr, kern), y_tr)
        print(f"  {label}:    {clf.score(kernel_matrix(X_te, X_tr, kern), y_te):.1%}")

compare_svm(0, 1, "Setosa vs Versicolor")
compare_svm(1, 2, "Versicolor vs Virginica")

Setosa vs Versicolor
  RBF kernel:          100.0%
  Quantum kernel A:    100.0%
  Quantum kernel B:    100.0%
Versicolor vs Virginica
  RBF kernel:          70.0%
  Quantum kernel A:    70.0%
  Quantum kernel B:    66.7%


## 6. Kết luận

- Setosa/Versicolor quá dễ: mọi kernel đều đạt 100%, không phân biệt được kernel nào tốt hơn.
- Versicolor/Virginica: các kernel cho kết quả tương đương nhau, **chưa thấy quantum advantage**. Điều này khớp với g/√N nhỏ ở phần 4.
- Bài học: một mạch trông "lượng tử" chưa chắc tạo ra kernel lượng tử thật sự. Cần kiểm tra cấu trúc mạch (như việc CNOT bị triệt tiêu) và đo bằng các chỉ số như geometric difference trước khi kỳ vọng lợi thế.